# Cross Validation — Practical Implementation

This notebook implements the techniques described in `notes.md`: K-Fold, Stratified K-Fold, and Leave-One-Out cross validation, followed by hyperparameter search with `GridSearchCV` and `RandomizedSearchCV`, all on the `load_breast_cancer` dataset.

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    KFold, StratifiedKFold, LeaveOneOut, cross_val_score,
    GridSearchCV, RandomizedSearchCV,
)
from sklearn.ensemble import RandomForestClassifier

data = load_breast_cancer()
X, y = data.data, data.target
print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Class balance: {np.bincount(y)}")


X shape: (569, 30), y shape: (569,)
Class balance: [212 357]


## K-Fold Cross Validation

Split the data into 5 folds; a `RandomForestClassifier` is trained on 4 folds and evaluated on the held-out fold, rotating through all 5 folds.

In [2]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
clf = RandomForestClassifier(n_estimators=100, random_state=42)

kfold_scores = cross_val_score(clf, X, y, cv=kf, scoring="accuracy")
print("K-Fold per-fold accuracy:", np.round(kfold_scores, 4))
print(f"K-Fold mean ± std: {kfold_scores.mean():.4f} ± {kfold_scores.std():.4f}")


K-Fold per-fold accuracy: [0.9561 0.9649 0.9386 0.9649 0.9646]
K-Fold mean ± std: 0.9578 ± 0.0102


## Stratified K-Fold Cross Validation

Same idea, but each fold preserves the overall class proportions — important since this is a classification task.

In [3]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

skf_scores = cross_val_score(clf, X, y, cv=skf, scoring="accuracy")
print("Stratified K-Fold per-fold accuracy:", np.round(skf_scores, 4))
print(f"Stratified K-Fold mean ± std: {skf_scores.mean():.4f} ± {skf_scores.std():.4f}")


Stratified K-Fold per-fold accuracy: [0.9649 0.9386 0.9561 0.9474 0.9735]
Stratified K-Fold mean ± std: 0.9561 ± 0.0123


## Leave-One-Out Cross Validation (LOOCV)

Each fold leaves out exactly one sample for testing. This gives `n_samples` folds — expensive, but uses the maximum possible amount of training data per fold. We report the mean and std of the (0/1) per-sample scores.

In [4]:
loo = LeaveOneOut()

# LOOCV is expensive (n_samples folds); this dataset has ~569 samples which is still tractable.
loo_scores = cross_val_score(clf, X, y, cv=loo, scoring="accuracy", n_jobs=-1)
print(f"LOOCV number of folds: {len(loo_scores)}")
print(f"LOOCV mean ± std: {loo_scores.mean():.4f} ± {loo_scores.std():.4f}")


LOOCV number of folds: 569
LOOCV mean ± std: 0.9613 ± 0.1928


## Comparing the three CV strategies

In [5]:
import pandas as pd

summary = pd.DataFrame({
    "method": ["K-Fold", "Stratified K-Fold", "Leave-One-Out"],
    "mean_accuracy": [kfold_scores.mean(), skf_scores.mean(), loo_scores.mean()],
    "std_accuracy": [kfold_scores.std(), skf_scores.std(), loo_scores.std()],
})
summary


,method,mean_accuracy,std_accuracy
0,K-Fold,0.957833,0.010188
1,Stratified K-Fold,0.956094,0.012340
2,Leave-One-Out,0.961336,0.192794


## Hyperparameter Search: GridSearchCV

`GridSearchCV` exhaustively tries every combination in a small hyperparameter grid for `RandomForestClassifier`, using Stratified K-Fold internally for classification tasks.

In [6]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)
grid_search.fit(X, y)

print("GridSearchCV best params:", grid_search.best_params_)
print(f"GridSearchCV best CV score: {grid_search.best_score_:.4f}")


GridSearchCV best params: {'max_depth': None, 'min_samples_leaf': 4, 'n_estimators': 50}
GridSearchCV best CV score: 0.9614


## Hyperparameter Search: RandomizedSearchCV

`RandomizedSearchCV` samples a fixed number of random combinations from the parameter distributions instead of trying all of them — much cheaper for larger search spaces.

In [7]:
from scipy.stats import randint

param_distributions = {
    "n_estimators": randint(50, 300),
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_leaf": randint(1, 6),
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_distributions,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
)
random_search.fit(X, y)

print("RandomizedSearchCV best params:", random_search.best_params_)
print(f"RandomizedSearchCV best CV score: {random_search.best_score_:.4f}")


RandomizedSearchCV best params: {'max_depth': None, 'min_samples_leaf': 3, 'n_estimators': 108}
RandomizedSearchCV best CV score: 0.9631


## Takeaway

- K-Fold and Stratified K-Fold give very similar mean accuracy here since the class imbalance in `load_breast_cancer` is mild, but Stratified K-Fold is the safer default for classification.
- LOOCV uses the most training data per fold and gives an almost unbiased estimate, but is far more expensive (as many folds as samples) and can have higher variance across individual fold scores.
- `GridSearchCV` and `RandomizedSearchCV` both search the same hyperparameter space; `RandomizedSearchCV` trades exhaustiveness for speed by sampling a fixed budget of combinations, which scales much better to larger grids.